# Week2_ex5 - Flux Linkage of Ring Coils Around a Straight Wire

This exercise looks at the H-field made by a long straight wire and checks how much of it links through two ring coils placed at different distances along the wire, using the EddyCurrent solver in Maxwell 3D.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil

Using the EddyCurrent solver here even though the physics is basically DC (straight wire carrying steady current) — mainly because Magnetostatic in 3D doesn't support the Radiation boundary, and EddyCurrent does. Frequency is set to a low 10Hz so skin effect stays negligible and the field distribution ends up close to the DC case anyway.

In [ ]:
# 1. open Ansys Electronics Desktop (AEDT)
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. turn off autosave
DT.disable_autosave()

# 3. create project and set solution type
sol_type = "EddyCurrent"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. get odesign object so recorded GUI scripts can be pasted directly
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##
proj_name = "Week2_ex5"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

## Save project and apply design name ##
proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)

desi_name = "Week2_ex5"
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()

In [ ]:
# function that finds the terminal faces of a coil-like object
# uses the center coordinate of each face and picks out the two faces with the largest |x| position
def find_terminal_face(winding_obj):
    terminal_face = []

    # find the maximum x position among all faces of the winding object
    max_x_pos = max(abs(face.center[0]) for face in winding_obj.faces)

    # collect the faces that sit at that max x position into the array
    for face in winding_obj.faces:
        if abs(abs(face.center[0]) - max_x_pos) <= 0.0001:
            terminal_face.append(face)

    # sort so that ter_out is the one with the larger x value
    ter_out, ter_in = sorted(terminal_face, key=lambda x: x.center[0], reverse=True)

    return ter_out, ter_in

In [ ]:
line_length = 400
M3D["line_length"] = f"{line_length}mm"

line_diameter = 10
M3D["line_diameter"] = f"{line_diameter}mm"

line_num_seg = 12
M3D["line_num_seg"] = f"{line_num_seg}"

Current = 100
M3D["Current"] = f"{Current}A"

In [ ]:
# 1. paste the setup script copied from GUI recording
oModule = oDesign.GetModule("AnalysisSetup")
oModule.InsertSetup("EddyCurrent", 
	[
		"NAME:Setup1",
		"Enabled:="		, True,
		[
			"NAME:MeshLink",
			"ImportMesh:="		, False
		],
		"MaximumPasses:="	, 10,
		"MinimumPasses:="	, 2,
		"MinimumConvergedPasses:=", 1,
		"PercentRefinement:="	, 15,
		"SolveFieldOnly:="	, False,
		"PercentError:="	, 1,
		"SolveMatrixAtLast:="	, True,
		"UseNonLinearIterNum:="	, False,
		"CacheSaveKind:="	, "Delta",
		"ConstantDelta:="	, "0s",
		"UseCacheFor:="		, ["Freq"],
		"UseIterativeSolver:="	, False,
		"RelativeResidual:="	, 1E-05,
		"NonLinearResidual:="	, 0.0001,
		"RelaxationFactor:="	, 1,
		"SmoothBHCurve:="	, True,
		"Frequency:="		, "10Hz",
		"HasSweepSetup:="	, False,
		"UseHighOrderShapeFunc:=", False,
		"ImportMeshForMuLink:="	, False,
		"LossAdaptiveCtrl:="	, "0.5",
		"UseMuLink:="		, False
	])

# 2. keep just the setup name as a string instead of grabbing the setup object.
# M3D.setups[-1] tries to look up the solution type internally, but AEDT 2025 R2
# renamed the EddyCurrent solver to "ac magnetic" and pyaedt 0.15.3's lookup table
# still only knows the old name, so it throws a KeyError. Just keeping the name
# string and calling oDesign.Analyze() directly later avoids this entirely.
setup_name = "Setup1"

In [ ]:
# 1. draw the straight wire
point_tmp = []
point_tmp.append(["-line_length/2", "0mm", "0mm"])
point_tmp.append(["line_length/2", "0mm", "0mm"])
line = M3D.modeler.create_polyline(points=point_tmp, name="line", material="copper",
                                    xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)


# 2. create region and assign radiation boundary condition
region_tmp = ["line_length/2", "-line_length/2", "line_length*2", "-line_length*2", "line_length*2", "-line_length*2"]
region = M3D.modeler.create_region(pad_value=region_tmp, pad_type="Absolute Position")

# NOTE: M3D.assign_radiation() checks that solution_type == "EddyCurrent" exactly and
# raises AEDTRuntimeError("Excitation applicable only to Eddy Current.") otherwise. But
# AEDT 2025 R2 renamed this solver "AC Magnetic" internally, so self.solution_type reports
# "AC Magnetic" here -- the check fails even though this IS an eddy current solve. Bypassing
# the pyaedt wrapper and calling the raw AEDT boundary API directly sidesteps this check.
oModule = oDesign.GetModule("BoundarySetup")
assignment = [region.top_face_y, region.bottom_face_y, region.top_face_z, region.bottom_face_z]
face_ids = [f.id for f in assignment]
oModule.AssignRadiation(
	[
		"NAME:Radiation1",
		"Objects:=", [],
		"Faces:=", face_ids
	])


# 3. create a dummy vacuum box to refine the mesh in the region we care about
box_origin = ["25mm", "-line_length*2", "-line_length/2"]
box_sizes = ["-2*25mm", "2*line_length", "line_length"]
dummy = M3D.modeler.create_box(origin=box_origin, sizes=box_sizes, name="dummy", material="vacuum")
M3D.modeler.subtract(blank_list=dummy, tool_list=line, keep_originals=True)


# 4. draw ring coil 1 (two cylinders, subtracted to leave a ring)
center = [0, "-line_length*(3/4)", "-1mm"]
origin = [0, "-line_length*(3/4)-20mm", "-1mm"]
ring_1 = M3D.modeler.create_polyhedron(orientation="Z", center=center, origin=origin, height="2mm", num_sides=36, name="ring_1", material="copper")

center = [0, "-line_length*(3/4)", "-1mm"]
origin = [0, "-line_length*(3/4)-18mm", "-1mm"]
tmp = M3D.modeler.create_polyhedron(orientation="Z", center=center, origin=origin, height="2mm", num_sides=36, name=None, material=None)
M3D.modeler.subtract(blank_list=ring_1, tool_list=tmp, keep_originals=False)

M3D.modeler.subtract(blank_list=dummy, tool_list=ring_1, keep_originals=True)


# 5. create a surface on ring coil 1 to pull the linked field data from later
origin = [0, "-line_length*(3/4)", 0]
field_circle_1 = M3D.modeler.create_circle(orientation="XY", origin=origin, radius="20mm", 
                                           num_sides=0, is_covered=True, name="field_circle_1", material=None, non_model=False)


# 6. draw ring coil 2, at twice the distance from the wire compared to ring 1
# (ring_1 sits at line_length*(3/4) = 300mm, ring_2 sits at line_length*(3/2) = 600mm,
# so if Ampere's law holds, coil2_phi should come out to roughly half of coil1_phi)
center = [0, "-line_length*(3/2)", "-1mm"]
origin = [0, "-line_length*(3/2)-20mm", "-1mm"]
ring_2 = M3D.modeler.create_polyhedron(orientation="Z", center=center, origin=origin, height="2mm", num_sides=36, name="ring_2", material="copper")

center = [0, "-line_length*(3/2)", "-1mm"]
origin = [0, "-line_length*(3/2)-18mm", "-1mm"]
tmp2 = M3D.modeler.create_polyhedron(orientation="Z", center=center, origin=origin, height="2mm", num_sides=36, name=None, material=None)
M3D.modeler.subtract(blank_list=ring_2, tool_list=tmp2, keep_originals=False)

M3D.modeler.subtract(blank_list=dummy, tool_list=ring_2, keep_originals=True)


# 7. create a surface on ring coil 2 to pull the linked field data from later
origin = [0, "-line_length*(3/2)", 0]
field_circle_2 = M3D.modeler.create_circle(orientation="XY", origin=origin, radius="20mm", 
                                           num_sides=0, is_covered=True, name="field_circle_2", material=None, non_model=False)

In [ ]:
oModule = oDesign.GetModule("MeshSetup")
oModule.AssignLengthOp(
	[
		"NAME:dummy_mesh",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["dummy"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "3000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "line_length/8",
	])

In [ ]:
# 1. set the terminal faces of the line as coil terminals
ter_out, ter_in = find_terminal_face(line)  # call the function that finds the coil's terminal faces

M3D.assign_coil(assignment=ter_out, conductors_number=1, polarity="Negative", name="Out")
M3D.assign_coil(assignment=ter_in, conductors_number=1, polarity="Positive", name="In")


# 2. create a winding and add the coils we just set up into it
coil = M3D.assign_winding(assignment=None, winding_type='Current', is_solid=True, current="Current", 
                        resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, 
                        name="coil")

M3D.add_winding_coils(coil.name, coils=["Out", "In"])

In [ ]:
oModule = oDesign.GetModule("FieldsReporter")

oModule.AddNamedExpression("coil1_phi", "Fields", 
	[
		"NameOfExpression:="	, ["Vector_B"],
		"Operation:="		, ["Normal"],
		"Operation:="		, ["Dot"],
		"EnterSurface:="	, ["field_circle_1"],
		"Operation:="		, ["SurfaceValue"],
		"Operation:="		, ["Integrate"],
	])
	
oModule.AddNamedExpression("coil2_phi", "Fields", 
	[
		"NameOfExpression:="	, ["Vector_B"],
		"Operation:="		, ["Normal"],
		"Operation:="		, ["Dot"],
		"EnterSurface:="	, ["field_circle_2"],
		"Operation:="		, ["SurfaceValue"],
		"Operation:="		, ["Integrate"]
	])

In [ ]:
# workaround for the same "ac magnetic" naming issue — oDesign.Analyze() is the
# raw AEDT API and skips pyaedt's solution-type lookup entirely
oDesign.Analyze(setup_name)

In [ ]:
oModule = oDesign.GetModule("ReportSetup")

oModule.CreateReport("Calculator Expressions Table 1", "Fields", "Data Table", "Setup1 : LastAdaptive", [], 
	[
		"Freq:="		, ["All"],
		"Phase:="		, ["0deg"],
		"line_length:="		, ["Nominal"],
		"line_diameter:="	, ["Nominal"],
		"line_num_seg:="	, ["Nominal"],
		"Current:="		, ["Nominal"]
	], 
	[
		"X Component:="		, "Freq",
		"Y Component:="		, ["coil1_phi","coil2_phi"]
	])

oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers", 
				"Calculator Expressions Table 1:coil1_phi:Phase=\'0deg\' [Curve1]"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Field Precision",
					"Value:="		, "10"
				]
			]
		]
	])
oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers", 
				"Calculator Expressions Table 1:coil2_phi:Phase=\'0deg\' [Curve1]"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Field Precision",
					"Value:="		, "10"
				]
			]
		]
	])

In [ ]:
oModule = oDesign.GetModule("FieldsReporter")

oModule.CreateFieldPlot(
	[
		"NAME:Mag_B1",
		"SolutionName:="	, "Setup1 : LastAdaptive",
		"UserSpecifyName:="	, 0,
		"UserSpecifyFolder:="	, 0,
		"QuantityName:="	, "Mag_B",
		"PlotFolder:="		, "B",
		"StreamlinePlot:="	, False,
		"AdjacentSidePlot:="	, False,
		"FullModelPlot:="	, False,
		"IntrinsicVar:="	, "Freq=\'10Hz\' Phase=\'0deg\'",
		"PlotGeomInfo:="	, [1,"Volume","ObjList",1,"dummy"],
		"FilterBoxes:="		, [0],
		[
			"NAME:PlotOnVolumeSettings",
			"PlotIsoSurface:="	, True,
			"PointSize:="		, 1,
			"Refinement:="		, 0,
			"CloudSpacing:="	, 0.5,
			"CloudMinSpacing:="	, -1,
			"CloudMaxSpacing:="	, -1,
			"ShadingType:="		, 0,
			"IsoMapTransparency:="	, True,
			"IsoTransparency:="	, 0.899999976158142,
			"IsoTransScaleThreshold:=", 0.200000002980232,
			[
				"NAME:Arrow3DSpacingSettings",
				"ArrowUniform:="	, True,
				"ArrowSpacing:="	, 0,
				"MinArrowSpacing:="	, 0,
				"MaxArrowSpacing:="	, 0
			]
		],
		"EnableGaussianSmoothing:=", False,
		"SurfaceOnly:="		, False
	], "Field")

oModule.SetPlotFolderSettings("B", 
	[
		"NAME:FieldsPlotSettings",
		"Real time mode:="	, True,
		[
			"NAME:ColorMapSettings",
			"ColorMapType:="	, "Spectrum",
			"SpectrumType:="	, "Rainbow",
			"UniformColor:="	, [127,255,255],
			"RampColor:="		, [255,127,127]
		],
		[
			"NAME:Scale3DSettings",
			"unit:="		, 104,
			"m_nLevels:="		, 10,
			"minvalue:="		, 1.9635E-05,
			"maxvalue:="		, 0.00025,
			"log:="			, False,
			"IntrinsicMin:="	, 1.96351322052549E-05,
			"IntrinsicMax:="	, 0.00438627410447518,
			"LimitFieldValuePrecision:=", False,
			"FieldValuePrecisionDigits:=", 4,
			"dB:="			, False,
			"AnimationStaticScale:=", False,
			"ScaleType:="		, 1,
			"UserSpecifyValues:="	, [11,1.96350002288818E-05,0.000456298919677734,0.000892962829589844,0.00132962670898437,0.00176629064941406,0.00220295458984375,0.00263961840820312,0.00307628247070312,0.0035129462890625,0.00394961010742187,0.00438627392578125],
		]
	])

In [ ]:
## Save project ##
M3D.save_project()